# Fase 0: fundamentos de redes neurais com NumPy
**Objetivo:** Compreender o funcionamento dos algoritmos fundamentais de redes neurais.

**Conceitos-chave:** regressão, gradiente descendente, neurônio, função de ativação, *backpropagation*.

### Sumário
*Etapa 0.1: Gradiente Descendente*

*Etapa 0.2: MLP do zero*

*Etapa 0.3: *Backpropagation**

**Etapa 0.4: *Autograd* simples**

*Mini-projeto 0: A curva de luz de uma supernova Ia*

In [1]:
import math

# 0.4 — *Autograd* simples

Agora que entendemos o mecanismo por trás de um MLP, precisamos generalizá-lo. Os códigos implementados são válidos apenas para *aquela* arquitetura específica, ou seja, duas camadas. Derivamos analiticamente a função de perda e, com base nisso, fizemos a *backpropagation*. Idealmente, desejamos que isso seja executado automaticamente, isto é, que não precisemos calcular, para cada nova arquitetura, as derivadas da função de perda. Quanto mais complexa a arquitetura, mais longas seriam as derivadas.

Para contornar este problema, iremos implementar um *autograd* simples. O *autograd* é o algoritmo que, além de armazenar o valor do gradiente (derivada) da função de perda, armazena também **como** essa derivada deve ser propagada para as camadas anteriores. Veremos isso com mais detalhes em seguida. A partir de agora, será comum o uso do termo **gradiente** no lugar de derivada, mas ambos são equivalentes nesse contexto.

Essa etapa é totalmente inspirada no código [micrograd](https://github.com/karpathy/micrograd/), implementado por Andrej Karpathy.

## 0.4.1 A classe `Valor`

Começaremos implementando uma classe chamada `Valor`. O papel dessa classe é armazenar um **valor** e seu **gradiente**.

In [2]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente deste valor.
    '''

    def __init__(self, valor):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0

Por enquanto, não podemos fazer muita coisa com essa classe, mas gostaríamos de, eventualmente, sermos capazes de realizar qualquer operação matemática usando essa própria classe, sem passar por valores numéricos ou *arrays* NumPy. Para isso, precisamos *implementar* as operações matemáticas mais fundamentais nesta classe. Começaremos pela **soma**. Iremos adicionar um método `__add__` na classe. O interpretador do Python, ao ver que a classe possui esse método, irá chamá-lo sempre que o símbolo `+` estiver atuando em objetos da classe `Valor`. O que desejamos é que, ao somar dois objetos `Valor`, o resultado seja também um objeto `Valor` cujo valor armazenado é o resultado da soma dos valores armazenados pelos outros dois objetos.

In [3]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente deste valor.
    '''

    def __init__(self, valor):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais"
        
        return soma

Vamos testar a classe.

In [4]:
a = Valor(1.0)
b = Valor(6.0)
c = a + b
print(f'A variável c é do tipo {type(c)}.')
print(f'O valor armazenado em c é {c.valor}.')

A variável c é do tipo <class '__main__.Valor'>.
O valor armazenado em c é 7.0.


Como a ideia principal é criar um algoritmo que seja capaz de realizar o *backpropagation* automaticamente, é importante mantermos um **registro** de quais valores geraram determinado valor. No exemplo acima, precisamos armazenar *no objeto* `c` que ele foi gerado a partir dos objetos `a` e `b`. Para isso, vamos atualizar a class `Valor`.

In [5]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais
        
        return soma
    
    def __repr__(self):
        return f"Valor(valor={self.valor}, grad={self.grad})"

Foi adicionado, também, o método `__repr__` para que, ao tentarmos imprimir um objeto `Valor`, ele tenha uma representação. Escrever `_pais` em vez de simplesmente `pais` é para indicar que essa vairável só deve ser acessada pelo **próprio** objeto, não devendo ser modificada por outros objetos do programa. Também é preciso modificar o método `__add__` para incluir o novo atributo da inicialização. Vamos testar novamente.

In [6]:
a = Valor(1.0)
b = Valor(6.0)
c = a + b
print(f'A variável c é do tipo {type(c)}.')
print(f'O valor armazenado em c é {c.valor}.')
print(f'O objeto c foi gerado a partir dos objetos {c._pais[0].valor} e {c._pais[1].valor}.')

A variável c é do tipo <class '__main__.Valor'>.
O valor armazenado em c é 7.0.
O objeto c foi gerado a partir dos objetos 1.0 e 6.0.


Vamos implementar outra operação fundamental: a multiplicação. A lógica será a mesma, mas aplicada ao método `__mul__`. Vamos atualizar a classe `Valor`.

In [7]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais
        
        return soma

    def __mul__(self, outro):
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        return prod

In [8]:
a = Valor(1.0)
b = Valor(6.0)
c = Valor(3.0)

d = c * b
e = a + d

print(f'd = {d.valor}\ne =  {e.valor}')

d = 18.0
e =  19.0


Agora, precisamos analisar como os gradientes de um valor se propagariam para os valores "pais". Por exemplo, como o gradiente de `e` impacta o gradiente de `a`? Vamos supoer que a função de perda $L$ é uma função de $e$. Então,
$$
\frac{\partial L}{\partial a} = \frac{d L}{d e} \frac{\partial e}{\partial a} = \frac{d L}{d e}.
$$
Ou seja, o gradiente de $e$ contribui com o próprio valor para o gradiente de $a$. Devido a isso, **somamos** o valor do gradiente de $e$ ao gradiente de $a$. A soma deve-se ao fato de que outras variáveis podem contribuir com seus gradientes para o gradiente de $a$, então devemos registrar o **acúmulo** dessas contribuições.

Vamos testar o mecanismo. Seja $L = e$. Então, temos que
$$
\frac{d L}{d e} = 1,
$$
e, portanto,
$$
\frac{\partial L}{\partial a} = \frac{d L}{d e} \frac{\partial e}{\partial a}.
$$
Como $e = a + d$, esperamos que o gradiente de $a$ seja
$$
\frac{\partial L}{\partial a} = \frac{d L}{d e} = 1.
$$
De forma análoga, é o resultado que esperamos para o gradiente de $d$. Vamos analisar agora $b$ e $c$:
$$
\frac{\partial L}{\partial b} = \frac{d L}{d e} \frac{\partial e}{\partial d} \frac{\partial d}{\partial b} = \frac{\partial L}{\partial d}\cdot c = 1\cdot 3 = 3.
$$
Analogamente, esperamos que, para $c$,
$$
\frac{\partial L}{\partial c} = \frac{\partial L}{\partial d}\cdot b = 1\cdot 6 = 6.
$$
Vamos ver se o mecanismo de propagação do gradiente funciona na prática.

In [9]:
a = Valor(1.0)
b = Valor(6.0)
c = Valor(3.0)

d = c * b
e = a + d
e.grad = 1.0

# Quando o "filho" surge da soma dos "pais", o gradiente dos pais é somado ao grandiente do filho
a.grad += e.grad
d.grad += e.grad

# Quando o "filho" surge da multiplicação dos "pais", o gradiente dos pais é somado ao gradiente do filho multiplicado pelo valor do outro pai
b.grad += d.grad * c.valor
c.grad += d.grad * b.valor

print(f'O gradiente de a é {a.grad}.')
print(f'O gradiente de d é {d.grad}.')
print(f'O gradiente de b é {b.grad}.')
print(f'O gradiente de c é {c.grad}.')

O gradiente de a é 1.0.
O gradiente de d é 1.0.
O gradiente de b é 3.0.
O gradiente de c é 6.0.


Os resultados foram o esperado.

Precisamos implementar isso na classe `Valor`. Cada *operação* deverá ter seu próprio método `.backward()` que distribuirá o gradiente do filho adequadamente aos pais. Vamos implementar.

In [10]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            self.outro += soma.grad
        
        return soma

    def __mul__(self, outro):
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        return prod

Ainda precisamos implementar o atributo `._backward` à classe `Valor`. Para cada objeto `Valor` gerado a partir das operações que implementamos, precisamos deixar registrado **qual** função `_backward()` aquele valor deve chamar quando precisar executar o *backpropagation*. Basta adicionarmos esse atributo na inicialização da classe e defini-lo ao final das operações.

In [11]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão
        self._backward = lambda: None

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            outro.grad += soma.grad

        soma._backward = _backward # Registra no objeto soma que a função que distribui os gradientes é essa _backward
        
        return soma

    def __mul__(self, outro):
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        prod._backward = _backward

        return prod

Com esse atributo e o método implementados, podemos realizar o *backpropagation* automaticamente, sem nos preocuparmos em definirmos os gradientes manualmente. Vamos repetir oe xemplo anterior.

In [12]:
a = Valor(1.0)
b = Valor(6.0)
c = Valor(3.0)

d = c * b
e = a + d
e.grad = 1.0

e._backward() # Passa o gradiente de e para os pais
d._backward() # Passa o gradiente de d para os pais

print(f'O gradiente de a é {a.grad}.')
print(f'O gradiente de d é {d.grad}.')
print(f'O gradiente de b é {b.grad}.')
print(f'O gradiente de c é {c.grad}.')

O gradiente de a é 1.0.
O gradiente de d é 1.0.
O gradiente de b é 3.0.
O gradiente de c é 6.0.


A implementação está funcional. Para tornar a propagação do gradiente automática de fato, precisamos implementar o método `backward()` global, isto é, o método que, ao chamarmos, chamará todos os outros `_backwards` na ordem correta, propagando os gradientes pela rede.

A ideia desse método é que construamos um **grafo** que conecta uma ponta da rede à outra. No nosso exemplo, um grafo que conecta a variável $e$ às variáveis $a$, $b$ e $c$. Para isso, construiremos uma lista contendo *todos* os elementos do grafo (todas as variáveis de $a$ até $e$) e percorreremos essa lista do fim para o início, chamando o método `_backward` de cada elemento.

Vamos implementar.

In [13]:
class Valor:
    '''Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão
        self._backward = lambda: None

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            outro.grad += soma.grad

        soma._backward = _backward # Registra no objeto soma que a função que distribui os gradientes é essa _backward
        
        return soma

    def __mul__(self, outro):
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        prod._backward = _backward

        return prod

    def backward(self):
        '''Método para montar o grafo da rede. Ele percorrerá todos os elementos da rede
        partindo de um nódulo inicial e irá seguir pela "linhagem" de um dos pais até que
        toda a linhagem esteja adicionada. Depois disso, percorrerá a outra linhagem até
        ela acabe também.
        '''
        grafo = [] # Lista em que salvaremos os elementos do grafo em ordem

        # Função recursiva que irá percorrer todo o grafo adicionando cada elemento do grafo à lista
        def montar_grafo(nodulo):
            if nodulo not in grafo: # Verifica se o nodulo em questão está na lista grafo, se não estiver, entra no if
                if nodulo._pais: # Verifica se o nodulo possui pais, se tiver, entra no if
                    for pai in nodulo._pais:
                        montar_grafo(pai) # Chama a própria função para cada pai do nódulo

                grafo.append(nodulo) # Se o nódulo não tiver pais, ou seja, for o fim de uma linhagem (início do grafo),
                                     # ele adiciona o nódulo na lista e começa a voltar no grafo adicionando cada um dos
                                     # outros nódulos filhos seguitnes.

        montar_grafo(self) # Chama a função que monta o grafo para partir do próprio objeto que chamou o método

        for nodulo in reversed(grafo):
            nodulo._backward() # Para nódulo na lista grafo invertida, chama o método _backward.

Repetindo o exemplo, podemos simplesmente chamar o método `.backward()` para o objeto $e$ que a função irá reconstruir todo o grafo e propagar os gradientes automaticamente.

In [14]:
a = Valor(1.0)
b = Valor(6.0)
c = Valor(3.0)

d = c * b
e = a + d
e.grad = 1.0

e.backward() # Monta o grafo e propaga os gradientes em ordem automaticamente

print(f'O gradiente de a é {a.grad}.')
print(f'O gradiente de d é {d.grad}.')
print(f'O gradiente de b é {b.grad}.')
print(f'O gradiente de c é {c.grad}.')

O gradiente de a é 1.0.
O gradiente de d é 1.0.
O gradiente de b é 3.0.
O gradiente de c é 6.0.


Obtemos, novamente, o mesmo resultado dos outros casos.

Vamos adicionar mais duas operações importantes: potênciação e a função ReLU. A lógica é mantida: criamos um novo objeto `Valor`, armazenamos seus pais e criamos o método `_backward` que deve ser chamado para propagar o gradiente daquela operação.

In [15]:
class Valor:
    ''' Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão
        self._backward = lambda: None

    def __add__(self, outro):
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            outro.grad += soma.grad

        soma._backward = _backward # Registra no objeto soma que a função que distribui os gradientes é essa _backward
        
        return soma

    def __mul__(self, outro):
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        prod._backward = _backward

        return prod

    def __pow__(self, outro):
        ''' Usualmente as potências são fixas, então estamos considerando, aqui
        apenas potências que são int ou float. Isso também evita um ValueError
        no gradiente de "outro" que surgiria se self.valor < 0.
        '''
        pot = Valor(self.valor ** outro, _pais=(self,))

        def _backward():
            self.grad += pot.grad * (outro * (self.valor ** (outro - 1)))

        pot._backward = _backward

        return pot

    def relu(self):
        z = Valor(0 if self.valor <= 0 else self.valor, _pais=(self,))

        def _backward():
            self.grad += z.grad * (z.valor > 0) # Se self.valor > 0, então o valor é 1. Senão, é 0.

        return z

    def backward(self):
        ''' Método para montar o grafo da rede. Ele percorrerá todos os elementos da rede
        partindo de um nódulo inicial e irá seguir pela "linhagem" de um dos pais até que
        toda a linhagem esteja adicionada. Depois disso, percorrerá a outra linhagem até
        ela acabe também.
        '''
        grafo = [] # Lista em que salvaremos os elementos do grafo em ordem

        # Função recursiva que irá percorrer todo o grafo adicionando cada elemento do grafo à lista
        def montar_grafo(nodulo):
            if nodulo not in grafo: # Verifica se o nodulo em questão está na lista grafo, se não estiver, entra no if
                if nodulo._pais: # Verifica se o nodulo possui pais, se tiver, entra no if
                    for pai in nodulo._pais:
                        montar_grafo(pai) # Chama a própria função para cada pai do nódulo

                grafo.append(nodulo) # Se o nódulo não tiver pais, ou seja, for o fim de uma linhagem (início do grafo),
                                     # ele adiciona o nódulo na lista e começa a voltar no grafo adicionando cada um dos
                                     # outros nódulos filhos seguitnes.

        montar_grafo(self) # Chama a função que monta o grafo para partir do próprio objeto que chamou o método

        for nodulo in reversed(grafo):
            nodulo._backward() # Para nódulo na lista grafo invertida, chama o método _backward.

    def __repr__(self):
        return f"Valor(valor={self.valor}, grad={self.grad})"

Com essas implementações, já é possível implementarmos uma rede neural. No entanto, para fins de completeza, vamos adicionar mais alguns detalhes seguindo o rigor do [micrograd](https://github.com/karpathy/micrograd/).

Primeiro, adicionaremos mecanismos para evitar erros de operação, por exemplo, se tentarmos realizar qualquer operação enter um `int` por um `Valor`, obteremos um erro, pois não há instruções de como realizar essas operações entre esses objetos. Para isso, adicionares uma simples linha em cada operação: `outro = outro if isinstance(outro, Valor) else Valor(outro)`.

In [16]:
class Valor:
    ''' Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão
        self._backward = lambda: None

    def __add__(self, outro):
        outro = outro if isinstance(outro, Valor) else Valor(outro) # Garante que a operação será executada entre dois objetos Valor, independente das entradas
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            outro.grad += soma.grad

        soma._backward = _backward # Registra no objeto soma que a função que distribui os gradientes é essa _backward
        
        return soma

    def __mul__(self, outro):
        outro = outro if isinstance(outro, Valor) else Valor(outro)
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        prod._backward = _backward

        return prod

    def __pow__(self, outro):
        ''' Usualmente as potências são fixas, então estamos considerando, aqui
        apenas potências que são int ou float. Isso também evita um ValueError
        no gradiente de "outro" que surgiria se self.valor < 0.
        '''
        outro = outro if isinstance(outro, Valor) else Valor(outro)
        pot = Valor(self.valor ** outro, _pais=(self,))

        def _backward():
            self.grad += pot.grad * (outro * (self.valor ** (outro - 1)))

        pot._backward = _backward

        return pot

    def relu(self):
        z = Valor(0 if self.valor <= 0 else self.valor, _pais=(self,))

        def _backward():
            self.grad += z.grad * (z.valor > 0) # Se self.valor > 0, então o valor é 1. Senão, é 0.

        return z

    def backward(self):
        ''' Método para montar o grafo da rede. Ele percorrerá todos os elementos da rede
        partindo de um nódulo inicial e irá seguir pela "linhagem" de um dos pais até que
        toda a linhagem esteja adicionada. Depois disso, percorrerá a outra linhagem até
        ela acabe também.
        '''
        grafo = [] # Lista em que salvaremos os elementos do grafo em ordem

        # Função recursiva que irá percorrer todo o grafo adicionando cada elemento do grafo à lista
        def montar_grafo(nodulo):
            if nodulo not in grafo: # Verifica se o nodulo em questão está na lista grafo, se não estiver, entra no if
                if nodulo._pais: # Verifica se o nodulo possui pais, se tiver, entra no if
                    for pai in nodulo._pais:
                        montar_grafo(pai) # Chama a própria função para cada pai do nódulo

                grafo.append(nodulo) # Se o nódulo não tiver pais, ou seja, for o fim de uma linhagem (início do grafo),
                                     # ele adiciona o nódulo na lista e começa a voltar no grafo adicionando cada um dos
                                     # outros nódulos filhos seguitnes.

        montar_grafo(self) # Chama a função que monta o grafo para partir do próprio objeto que chamou o método

        for nodulo in reversed(grafo):
            nodulo._backward() # Para nódulo na lista grafo invertida, chama o método _backward.

    def __repr__(self):
        return f"Valor(valor={self.valor}, grad={self.grad})"

Quando escrevemos `a + b`, o que o Python faz é `a.__add__(a, b)` e, se isso falhar, ele faz `b.__radd__(b, a)`, ou seja, chama o método de soma de `b` quando `b` está na direita. Isso evita possíveis erros de operações entre objetos de classes distintas. Vamos implementar isso *juntamente* com o negativo de um `Valor`, a operação de subtração, a operação de divisão e as suas versões à direita.

In [17]:
class Valor:
    ''' Armazena um valor e seu gradiente.

    Atributos:
        valor: o número a ser armazenado.
        grad: o gradiente da função de perda em relação a este valor.
    '''

    def __init__(self, valor, _pais=()):
        self.valor = valor # O número que será armazenado
        self.grad = 0.0 # O gradiente deste valor, inicializado como 0
        self._pais = _pais # Tupla que irá armazenar quais objetos geraram o objeto em questão
        self._backward = lambda: None

    def __add__(self, outro):
        outro = outro if isinstance(outro, Valor) else Valor(outro) # Garante que a operação será executada entre dois objetos Valor, independente das entradas
        soma = Valor(self.valor + outro.valor, _pais=(self, outro)) # Cria um objeto Valor a partir da soma dos valores armazenados pelos objetos "pais" e armazena os valores pais

        # Método que, quando chamado, propagará o gradiente pelos pais do objeto.
        def _backward():
            self.grad += soma.grad
            outro.grad += soma.grad

        soma._backward = _backward # Registra no objeto soma que a função que distribui os gradientes é essa _backward
        
        return soma

    def __mul__(self, outro):
        outro = outro if isinstance(outro, Valor) else Valor(outro)
        prod = Valor(self.valor * outro.valor, _pais=(self, outro))  # Cria um objeto Valor a partir da multiplicação dos valores armazenados pelos objetos "pais" e armazena os valores pais

        def _backward():
            self.grad += prod.grad * outro.valor
            outro.grad += prod.grad * self.valor

        prod._backward = _backward

        return prod

    def __pow__(self, outro):
        ''' Usualmente as potências são fixas, então estamos considerando, aqui
        apenas potências que são int ou float. Isso também evita um ValueError
        no gradiente de "outro" que surgiria se self.valor < 0.
        '''
        pot = Valor(self.valor ** outro, _pais=(self,))

        def _backward():
            self.grad += pot.grad * (outro * (self.valor ** (outro - 1)))

        pot._backward = _backward

        return pot

    def relu(self):
        z = Valor(0 if self.valor <= 0 else self.valor, _pais=(self,))

        def _backward():
            self.grad += z.grad * (z.valor > 0) # Se self.valor > 0, então o valor é 1. Senão, é 0.

        return z

    def backward(self):
        ''' Método para montar o grafo da rede. Ele percorrerá todos os elementos da rede
        partindo de um nódulo inicial e irá seguir pela "linhagem" de um dos pais até que
        toda a linhagem esteja adicionada. Depois disso, percorrerá a outra linhagem até
        ela acabe também.
        '''
        grafo = [] # Lista em que salvaremos os elementos do grafo em ordem

        # Função recursiva que irá percorrer todo o grafo adicionando cada elemento do grafo à lista
        def montar_grafo(nodulo):
            if nodulo not in grafo: # Verifica se o nodulo em questão está na lista grafo, se não estiver, entra no if
                if nodulo._pais: # Verifica se o nodulo possui pais, se tiver, entra no if
                    for pai in nodulo._pais:
                        montar_grafo(pai) # Chama a própria função para cada pai do nódulo

                grafo.append(nodulo) # Se o nódulo não tiver pais, ou seja, for o fim de uma linhagem (início do grafo),
                                     # ele adiciona o nódulo na lista e começa a voltar no grafo adicionando cada um dos
                                     # outros nódulos filhos seguitnes.

        montar_grafo(self) # Chama a função que monta o grafo para partir do próprio objeto que chamou o método

        for nodulo in reversed(grafo):
            nodulo._backward() # Para nódulo na lista grafo invertida, chama o método _backward.

    def __radd__(self, outro): # Instrui como realizar a operação outro + self
        return self + outro

    def __rmul__(self, outro):
        return self * outro

    def __neg__(self):
        return -1 * self        
    
    def __sub__(self, outro):
        return self + (-outro)

    def __rsub__(self, outro):
        return outro + (-self)

    def __truediv__(self, outro):
        return self * outro ** -1

    def __rtruediv__(self, outro):
        return outro * self ** -1

    def __repr__(self):
        return f"Valor(valor={self.valor}, grad={self.grad})"

Agora, nossa classe `Valor` é capaz de executar *todas* as operações básica e, consequentemente, operações mais complexas derivadas dessas mais básicas.

Assim, finalizamos os **fundamentos** de redes neurais e do funcionamento de um *autograd*. Nas fases futuras, esses conceitos se mostrarão muito úteis para entendermos como o `PyTorch` atua por de baixo dos panos.